In [3]:
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.datasets import make_classification, make_regression
from sklearn.metrics import accuracy_score, mean_squared_error
from custom.util.data_manipulation import load_and_process_data
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from custom.models.decision_tree import DecisionTree



X, y, numeric_columns, categorical_columns = load_and_process_data("C:/Users/barte/Desktop/ITU-MachineLearning-FinalProject-Couriers/data/claims_train.csv", False, None )

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
    ]
)


pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("tree", DecisionTree()),
    ]
)

param_grid = {
    "tree__max_depth": [None, 5, 10, 16],
    "tree__max_leaves": [2, 10, 50, 100], # Note: max_leaves=1 will likely crash or result in no splits
    "tree__min_to_split": [2, 10, 20],     # Changed from min_samples_split
    "tree__min_samples_leaf": [1, 2, 5, 0.05],
    "tree__min_impurity_decrease": [0.0, 0.01, 0.1],
}

grid = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_grid,
    n_iter=50, 
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)

X_small = X.sample(n=50_000, random_state=42)
y_small = y.loc[X_small.index]


grid.fit(X_small, y_small)



print("Best parameters:")
print(grid.best_params_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X)


c:\Users\barte\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:1102: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


Best parameters:
{'tree__min_to_split': 10, 'tree__min_samples_leaf': 0.05, 'tree__min_impurity_decrease': 0.01, 'tree__max_leaves': 100, 'tree__max_depth': None}


TypeError: 'int' object is not subscriptable